In [1]:
# imports for my testing
import torch
import tiktoken

In [2]:
# initializing model and loading weights
from gpt_model import GPTModel
from config import GPTConfig
from transformers.generation.utils import GenerationMixin

# moving to cuda if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device being used: {device}")

# Patch GenerationMixin to ensure the model supports .generate() function
if GenerationMixin not in GPTModel.__bases__:
    print("injecting GenerationMixin into base GPTModel structural layers...")
    GPTModel.__bases__ = (GPTModel.__bases__[0], GenerationMixin) + GPTModel.__bases__[1:]

#setting-up configurations
config = GPTConfig()
model = GPTModel(config)

#loading weights
final_weights_pth = "chat_title_generator_model.pth"
state_dict = torch.load(final_weights_pth, map_location=device)

model.load_state_dict(state_dict)
model.to(device)
model.eval()


/home/toqeer-yasir/miniconda3/envs/lg/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device being used: cpu
injecting GenerationMixin into base GPTModel structural layers...


GPTModel(
  (tok_emb): Embedding(50257, 1024)
  (pos_emb): Embedding(1024, 1024)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (dropout): Dropout(p=0.1, inplace=False)
        (W_query): Linear(in_features=1024, out_features=1024, bias=True)
        (W_key): Linear(in_features=1024, out_features=1024, bias=True)
        (W_value): Linear(in_features=1024, out_features=1024, bias=True)
        (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=1024, out_features=4096, bias=True)
          (1): GELU()
          (2): Linear(in_features=4096, out_features=1024, bias=True)
        )
      )
      (norm1): LayerNormalization()
      (norm2): LayerNormalization()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (

In [3]:
tokenizer = tiktoken.get_encoding("gpt2")

def predict_chat_title(chat_transcript: str, max_tokens: int = 15) -> str:
    """
    Constructs the prompt engineering format, matches training formatting conditions,
    and runs iterative token generation on the fine-tuned model.
    """
    # setting up my previous training prompt
    transcript_lower = chat_transcript.lower()
    coding_keywords = ["how", "write", "code", "delete", "run", "fix"]
    
    if any(keyword in transcript_lower for keyword in coding_keywords):
        instruction = "Read the following chat log and provide a short title describing what the user wants or what is being discussed."
    else:
        instruction = "Read the following conversation and generate a short declarative title summarizing its core conceptual theme."

    # Build prompt structure identically to DatasetProcessing constraints
    formatted_prompt = (
        f"### Instruction:\n{instruction}\n\n"
        f"### Conversation:\n{chat_transcript.strip()}\n\n"
        f"### Title:\n"
    )

    #text to input tensors
    input_tokens = tokenizer.encode(formatted_prompt)
    input_tensor = torch.tensor([input_tokens]).to(device)
    
    generated_sequence = []

# predicting tokens
    with torch.no_grad():
        for _ in range(max_tokens):
            model_outputs = model(input_tensor)
            
            # Handle output parsing variations (tuple returns vs pure tensors)
            logits = model_outputs[1] if isinstance(model_outputs, tuple) else model_outputs
            
            # Greedy search decoding step: choose highest confidence next token
            next_token_id = torch.argmax(logits[:, -1, :], dim=-1).item()
            
            # Stop generating if the End-Of-Text token is reached
            if next_token_id == tokenizer.eot_token:
                break
                
            generated_sequence.append(next_token_id)
            
            # Append new token back to context array for next loop step
            new_token_tensor = torch.tensor([[next_token_id]]).to(device)
            input_tensor = torch.cat([input_tensor, new_token_tensor], dim=1)

    #decoding & cleanning-up
    predicted_title = tokenizer.decode(generated_sequence)
    cleaned_title = predicted_title.split("\n")[0].replace("###", "").strip(".,!? ")
    
    return cleaned_title


In [4]:
example = """
"category": "General Conceptual Theme",
"chat": '''
user: I've been feeling extremely overwhelmed by my workload this week. There are too many competing deadlines.
assistant: Don't worry. I'm here to help you out can you provide me full details so i can solve some of your problems?
'''
"""

title_prediction = predict_chat_title(example)
print(f"prediction output: {title_prediction}")

prediction output: Don't worry. I'm here to help you out can you provide me
